# Predikcija F1 rezultata 2025-2026

Notebook je pregled rezultata koje generiŠe `run_analysis.py`. Za potpunu reprodukciju prvo pokrenuti skriptu iz glavnog direktorija projekta.

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUT = ROOT / "outputs"
TABLES = OUT / "tables"
print(ROOT)

/mnt/data/F1_seminarski_projekat


## Opcionalno: ponovno pokretanje cijelog procesa

Ukloniti `#` samo kada želite ponovo trenirati model i izvršiti simulacije.

In [2]:
# import subprocess, sys
# subprocess.run([sys.executable, str(ROOT / "run_analysis.py"), "--project-root", str(ROOT)], check=True)

## Sažetak skupa podataka

In [3]:
inventory = pd.read_csv(TABLES / "dataset_inventory.csv")
inventory[["file", "rows", "columns", "min_year", "max_year"]]

,file,rows,columns,min_year,max_year
0,circuits.csv,77,9,NaN,NaN
1,constructor_results.csv,12625,5,NaN,NaN
2,constructor_standings.csv,13391,7,NaN,NaN
3,constructors.csv,212,5,NaN,NaN
4,driver_standings.csv,34863,7,NaN,NaN
5,drivers.csv,861,9,NaN,NaN
6,lap_times.csv,589081,6,NaN,NaN
7,pit_stops.csv,11371,7,NaN,NaN
8,qualifying.csv,10494,9,NaN,NaN
9,races.csv,1125,18,1950.0,2024.0


In [4]:
quality = json.loads((TABLES / "data_quality_summary.json").read_text())
pd.Series(quality, name="value").to_frame()

,value
source_start_year,1950
source_end_year,2024
source_races,1125
source_result_rows,26759
source_drivers,861
source_constructors,211
duplicate_driver_race_rows,91
missing_finish_order,0
missing_driver_ref,0
missing_constructor_ref,0


## HronoloŠka validacija 2019-2024

In [5]:
backtest = pd.read_csv(TABLES / "backtest_by_year.csv")
backtest[["test_year", "model_rank_mae", "baseline_rank_mae", "model_winner_accuracy", "baseline_winner_accuracy"]]

,test_year,model_rank_mae,baseline_rank_mae,model_winner_accuracy,baseline_winner_accuracy
0,2019,3.861905,3.676190,0.285714,0.523810
1,2020,3.941176,3.947059,0.647059,0.647059
2,2021,3.518182,3.531818,0.363636,0.318182
3,2022,3.840909,3.731818,0.500000,0.500000
4,2023,3.731818,3.731818,0.727273,0.863636
5,2024,3.678497,3.640919,0.375000,0.333333


## Prognoza 2025. naspram stvarnog konačnog poretka

In [6]:
comparison_2025 = pd.read_csv(TABLES / "comparison_2025_driver_standings.csv")
comparison_2025[["driver_name", "predicted_position", "actual_position", "predicted_points", "actual_points", "absolute_rank_error"]].head(15)

,driver_name,predicted_position,actual_position,predicted_points,actual_points,absolute_rank_error
0,Lando Norris,1,1,379.941667,423,0
1,Max Verstappen,4,2,302.932000,421,2
2,Oscar Piastri,2,3,343.624333,410,1
3,George Russell,5,4,269.471000,319,1
4,Charles Leclerc,3,5,331.204667,242,2
5,Lewis Hamilton,6,6,245.894000,156,0
6,Kimi Antonelli,7,7,139.596000,150,0
7,Alexander Albon,20,8,20.651333,73,12
8,Carlos Sainz,11,9,60.776333,64,2
9,Fernando Alonso,9,10,76.674667,56,1


## Predsezonska prognoza 2026. naspram stanja nakon 13. utrke

In [7]:
comparison_2026 = pd.read_csv(TABLES / "comparison_2026_preseason_driver_standings_to_round13.csv")
comparison_2026[["driver_name", "predicted_position", "actual_position", "predicted_points", "actual_points", "absolute_rank_error"]].head(15)

,driver_name,predicted_position,actual_position,predicted_points,actual_points,absolute_rank_error
0,Kimi Antonelli,6.0,1,132.994333,267,5.0
1,George Russell,4.0,2,155.655667,201,2.0
2,Lewis Hamilton,7.0,3,125.105000,191,4.0
3,Lando Norris,1.0,4,230.062333,171,3.0
4,Charles Leclerc,5.0,5,150.545000,155,0.0
5,Max Verstappen,3.0,6,169.666667,127,3.0
6,Oscar Piastri,2.0,7,217.777333,116,5.0
7,Isack Hadjar,8.0,8,74.340333,71,0.0
8,Liam Lawson,13.0,9,17.510667,51,4.0
9,Pierre Gasly,19.0,10,11.159333,41,9.0


## Ažurirana projekcija konačnog poretka 2026.

In [8]:
nowcast = pd.read_csv(TABLES / "nowcast_2026_after_round13_driver_standings.csv")
nowcast[["predicted_position", "driver_name", "expected_points", "points_p10", "points_p90", "championship_probability"]].head(15)

,predicted_position,driver_name,expected_points,points_p10,points_p90,championship_probability
0,1,Kimi Antonelli,417.134667,382.0,451.0,0.961667
1,2,George Russell,346.643000,311.0,382.0,0.036000
2,3,Lewis Hamilton,297.514333,262.0,333.0,0.001667
3,4,Lando Norris,287.713000,254.0,322.0,0.000667
4,5,Charles Leclerc,276.411000,242.0,312.0,0.000000
5,6,Oscar Piastri,213.848000,182.0,246.0,0.000000
6,7,Max Verstappen,208.158000,178.0,239.0,0.000000
7,8,Isack Hadjar,135.609667,107.0,166.0,0.000000
8,9,Liam Lawson,81.537667,63.0,102.1,0.000000
9,10,Pierre Gasly,67.720333,50.0,89.0,0.000000


## Kandidati za pobjede u preostalim utrkama

In [9]:
wins = pd.read_csv(TABLES / "nowcast_2026_remaining_top3_win_candidates.csv")
wins[["round", "race_name", "driver_name", "win_probability"]]

,round,race_name,driver_name,win_probability
0,14,Spanish Grand Prix,Kimi Antonelli,0.237000
1,14,Spanish Grand Prix,George Russell,0.217667
2,14,Spanish Grand Prix,Charles Leclerc,0.150000
3,15,Azerbaijan Grand Prix,Kimi Antonelli,0.245333
4,15,Azerbaijan Grand Prix,George Russell,0.204667
5,15,Azerbaijan Grand Prix,Charles Leclerc,0.132333
6,16,Bahrain Grand Prix in Malaysia,Kimi Antonelli,0.242000
7,16,Bahrain Grand Prix in Malaysia,George Russell,0.199667
8,16,Bahrain Grand Prix in Malaysia,Charles Leclerc,0.138333
9,17,Singapore Grand Prix,Kimi Antonelli,0.224333
